# wandb-finish — faded example 2: Place wandb.finish() in a finally block

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-finish`. Running the beacon reports progress on the `Logging: wandb.finish` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.finish` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-finish`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-finish"
DD_SUBTOPIC = "Logging: wandb.finish"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To guarantee `wandb.finish()` runs even if the training loop raises an exception, place it in a `finally` block. Python always executes `finally` code before propagating any exception, so the wandb run is properly closed regardless of whether training succeeds or crashes.

## Faded exercise 2

Implement `safe_train_fn(n_steps, raise_at)` so that `wandb.finish()` is always called — both on normal completion and on exception.

1. Call `wandb.init(project='proj')`.
2. Wrap the training loop in `try:` ... `finally:`.
3. Inside the loop, raise `RuntimeError` if `step == raise_at` (and `raise_at is not None`).
4. Complete the blank to place `wandb.finish()` in the `finally` clause.
5. Return `completed` (steps done before any raise).

**Fill in:** Call wandb.finish() inside the finally block so it runs regardless of whether the try block raised.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def safe_train_fn(n_steps, raise_at=None):
    wandb.init(project='proj')
    completed = 0
    try:
        for step in range(n_steps):
            if raise_at is not None and step == raise_at:
                raise RuntimeError(f'fail at {step}')
            completed += 1
    finally:
        wandb.finish()
    return completed

wandb.finish.reset_mock()
safe_train_fn(3)
print('finish called:', wandb.finish.call_count)


import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def safe_train_fn(n_steps, raise_at=None):
    wandb.init(project='proj')
    completed = 0
    try:
        for step in range(n_steps):
            if raise_at is not None and step == raise_at:
                raise RuntimeError(f'fail at {step}')
            completed += 1
    finally:
        wandb.finish()
    return completed

def _test():
    # Normal path
    wandb.init.reset_mock(); wandb.finish.reset_mock()
    r = safe_train_fn(5)
    assert r == 5
    assert wandb.finish.call_count == 1
    # Exception path
    wandb.init.reset_mock(); wandb.finish.reset_mock()
    try:
        safe_train_fn(5, raise_at=2)
        assert False, 'should have raised'
    except RuntimeError:
        pass
    assert wandb.finish.call_count == 1, 'finish must run even on exception'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def safe_train_fn(n_steps, raise_at=None):
    wandb.init(project='proj')
    completed = 0
    try:
        for step in range(n_steps):
            if raise_at is not None and step == raise_at:
                raise RuntimeError(f'fail at {step}')
            completed += 1
    finally:
        wandb.finish()
    return completed

wandb.finish.reset_mock()
safe_train_fn(3)
print('finish called:', wandb.finish.call_count)
```
</details>